In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

print("YOLO is working! 🚗🤖")

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
from IPython.display import Video, display

video_path = next(iter(uploaded))

print("🎥 Original CCTV Video")
print("File:", video_path)

display(Video(video_path, embed=True))

In [ ]:
results = model.predict(
    source=next(iter(uploaded)),
    save=True,
    conf=0.25
)

In [ ]:
import os
from IPython.display import Video

# Get the directory where results were saved by YOLO
# results is a list of Results objects, take the first one
output_folder = results[0].save_dir

# Get the original filename from the uploaded dictionary
# uploaded is a dictionary where keys are filenames
video_input_filename = next(iter(uploaded))

# Assume YOLO saves the processed video with the same name in the save_dir
video_file = video_input_filename

Video(os.path.join(output_folder, video_file), embed=True)

In [ ]:
import os

print(os.listdir("/content/runs/detect/predict"))

In [ ]:
from IPython.display import Video

Video(
    "/content/runs/detect/predict/6420_India_Traffic_1280x720 (1).avi",
    embed=True
)

In [ ]:
import cv2

input_video = "/content/runs/detect/predict/6420_India_Traffic_1280x720 (1).avi"
output_video = "/content/traffic_result.mp4"

cap = cv2.VideoCapture(input_video)

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

while True:
    ret, frame = cap.read()
    if not ret:
        break
    out.write(frame)

cap.release()
out.release()

print("Converted successfully! ✅")

In [ ]:
from IPython.display import Video

Video("/content/traffic_result.mp4", embed=True)

In [ ]:
import cv2

cap = cv2.VideoCapture("/content/traffic_result.mp4")

print("Opened:", cap.isOpened())
print("FPS:", cap.get(cv2.CAP_PROP_FPS))
print("Frames:", int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
print("Width:", int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)))
print("Height:", int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))

cap.release()

In [ ]:
import cv2
from IPython.display import display
from PIL import Image

cap = cv2.VideoCapture("/content/traffic_result.mp4")

ret, frame = cap.read()
cap.release()

if ret:
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    display(Image.fromarray(frame))
else:
    print("Could not read the video ❌")

In [ ]:
results = model.track(
    source="/content/6420_India_Traffic_1280x720 (1).avi",
    tracker="bytetrack.yaml",
    save=True,
    conf=0.25
)

In [ ]:
import os

print(os.listdir("/content"))

In [ ]:
results = model.track(
    source="/content/6420_India_Traffic_1280x720 (1).mp4",
    tracker="bytetrack.yaml",
    save=True,
    conf=0.25
)

In [ ]:
import os

print(os.listdir("/content/runs/detect/predict"))

In [ ]:
from IPython.display import Video

Video(
    "/content/runs/detect/predict/6420_India_Traffic_1280x720 (1).avi",
    embed=True
)

In [ ]:
import cv2

input_video = "/content/runs/detect/predict/6420_India_Traffic_1280x720 (1).avi"
output_video = "/content/tracking_result.mp4"

cap = cv2.VideoCapture(input_video)

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

while True:
    ret, frame = cap.read()
    if not ret:
        break
    out.write(frame)

cap.release()
out.release()

print("Tracking video converted to MP4! ✅")

In [ ]:
from IPython.display import Video

Video("/content/tracking_result.mp4", embed=True)

In [ ]:
import cv2
from IPython.display import display
from PIL import Image

video = "/content/tracking_result.mp4"

cap = cv2.VideoCapture(video)

# Jump to frame 100
cap.set(cv2.CAP_PROP_POS_FRAMES, 100)

ret, frame = cap.read()
cap.release()

if ret:
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    display(Image.fromarray(frame))
    print("Frame loaded successfully ✅")
else:
    print("Could not read frame ❌")

In [ ]:
from collections import Counter

vehicle_counts = Counter()

for result in results:
    if result.boxes is not None and result.boxes.id is not None:
        classes = result.boxes.cls.cpu().numpy().astype(int)

        for cls in classes:
            vehicle_counts[model.names[cls]] += 1

print("Vehicle detections:")
for vehicle, count in vehicle_counts.items():
    print(f"{vehicle}: {count}")

In [ ]:
from collections import defaultdict

unique_vehicles = defaultdict(set)

vehicle_classes = {
    2: "car",
    3: "motorcycle",
    5: "bus",
    7: "truck"
}

for result in results:
    if result.boxes is not None and result.boxes.id is not None:
        track_ids = result.boxes.id.cpu().numpy().astype(int)
        classes = result.boxes.cls.cpu().numpy().astype(int)

        for track_id, cls in zip(track_ids, classes):
            if cls in vehicle_classes:
                unique_vehicles[vehicle_classes[cls]].add(track_id)

print("UNIQUE VEHICLES 🚦")
for vehicle, ids in unique_vehicles.items():
    print(f"{vehicle}: {len(ids)}")

print("\nTotal unique vehicles:",
      sum(len(ids) for ids in unique_vehicles.values()))

In [ ]:
total_vehicles = sum(len(ids) for ids in unique_vehicles.values())

if total_vehicles < 30:
    traffic_level = "LOW 🟢"
elif total_vehicles < 60:
    traffic_level = "MEDIUM 🟡"
else:
    traffic_level = "HIGH 🔴"

print("🚦 TRAFFIC ANALYSIS")
print("Total unique vehicles:", total_vehicles)
print("Traffic level:", traffic_level)

In [ ]:
vehicle_density = total_vehicles / 6.4

print("🚦 TRAFFIC DENSITY")
print("Vehicles per second:", round(vehicle_density, 2))

In [ ]:
print("🚦 VEHICLE TYPE BREAKDOWN")

for vehicle, ids in unique_vehicles.items():
    print(f"{vehicle.capitalize()}: {len(ids)}")

In [ ]:
import matplotlib.pyplot as plt

vehicles = list(unique_vehicles.keys())
counts = [len(ids) for ids in unique_vehicles.values()]

plt.figure(figsize=(8, 5))
plt.bar(vehicles, counts)
plt.title("Vehicle Type Distribution")
plt.xlabel("Vehicle Type")
plt.ylabel("Number of Vehicles")
plt.show()

In [ ]:
traffic_data = {
    "camera_id": "CAM_01",
    "total_vehicles": total_vehicles,
    "traffic_level": traffic_level,
    "vehicle_density": round(vehicle_density, 2),
    "cars": len(unique_vehicles["car"]),
    "motorcycles": len(unique_vehicles["motorcycle"]),
    "buses": len(unique_vehicles["bus"]),
    "trucks": len(unique_vehicles["truck"])
}

print("📡 CAMERA TRAFFIC DATA")
print(traffic_data)

In [ ]:
camera_data = [
    traffic_data,

    {
        "camera_id": "CAM_02",
        "total_vehicles": 52,
        "traffic_level": "MEDIUM 🟡",
        "vehicle_density": 8.13,
        "cars": 28,
        "motorcycles": 15,
        "buses": 3,
        "trucks": 6
    },

    {
        "camera_id": "CAM_03",
        "total_vehicles": 21,
        "traffic_level": "LOW 🟢",
        "vehicle_density": 3.28,
        "cars": 12,
        "motorcycles": 6,
        "buses": 1,
        "trucks": 2
    }
]

print("🏙️ CITY-WIDE TRAFFIC")
print()

for camera in camera_data:
    print(
        camera["camera_id"],
        "→",
        camera["traffic_level"],
        "| Vehicles:",
        camera["total_vehicles"]
    )

In [ ]:
import matplotlib.pyplot as plt

camera_ids = [camera["camera_id"] for camera in camera_data]
vehicle_counts = [camera["total_vehicles"] for camera in camera_data]

plt.figure(figsize=(8, 5))
plt.bar(camera_ids, vehicle_counts)

plt.title("City-Wide Traffic by Camera")
plt.xlabel("Camera")
plt.ylabel("Number of Vehicles")

plt.show()

In [ ]:
import pandas as pd

df = pd.DataFrame(camera_data)

print("🚦 INTELLIGENT TRAFFIC MONITOR")
display(df)

In [ ]:
%%writefile app.py

import streamlit as st

st.set_page_config(
    page_title="Intelligent Traffic Monitor",
    page_icon="🚦",
    layout="wide"
)

st.title("🚦 Intelligent Traffic Monitoring System")
st.write("AI-powered multi-camera traffic analysis")

st.subheader("🏙️ City-Wide Traffic")

col1, col2, col3 = st.columns(3)

with col1:
    st.metric("CAM_01 Vehicles", 83)
    st.error("🔴 HIGH TRAFFIC")

with col2:
    st.metric("CAM_02 Vehicles", 52)
    st.warning("🟡 MEDIUM TRAFFIC")

with col3:
    st.metric("CAM_03 Vehicles", 21)
    st.success("🟢 LOW TRAFFIC")

st.subheader("📊 Vehicle Distribution")

st.write("🚗 Cars: 75")
st.write("🏍️ Motorcycles: 46")
st.write("🚌 Buses: 8")
st.write("🚚 Trucks: 27")

In [ ]:
!pip install streamlit pyngrok -q

In [ ]:
!streamlit run app.py --server.port 8501 &>/content/log.txt &

In [ ]:
print(open("/content/log.txt").read())

In [ ]:
!streamlit run app.py --server.port 8502 &>/content/log2.txt &

In [ ]:
print(open("/content/log2.txt").read())

In [ ]:
from google.colab import output

output.serve_kernel_port_as_window(8502)

In [ ]:
!curl -I http://localhost:8502

In [ ]:
from google.colab import output

output.serve_kernel_port_as_iframe(8502)

In [ ]:
!streamlit run app.py --server.port 8503 --server.address 0.0.0.0 --server.enableCORS false --server.enableXsrfProtection false &>/content/log3.txt &

In [ ]:
print(open("/content/log3.txt").read())

In [ ]:
!pkill -f streamlit

In [ ]:
import pandas as pd

dashboard = df[
    [
        "camera_id",
        "total_vehicles",
        "traffic_level",
        "vehicle_density",
        "cars",
        "motorcycles",
        "buses",
        "trucks"
    ]
]

display(dashboard)

In [ ]:
print("🚨 TRAFFIC ALERTS")
print("=" * 40)

for camera in camera_data:
    if "HIGH" in camera["traffic_level"]:
        print(
            f"🚨 ALERT: {camera['camera_id']} has HIGH traffic!"
        )
    elif "MEDIUM" in camera["traffic_level"]:
        print(
            f"⚠️ WARNING: {camera['camera_id']} has MEDIUM traffic."
        )
    else:
        print(
            f"✅ CLEAR: {camera['camera_id']} has LOW traffic."
        )

In [ ]:
print("🧠 CONGESTION ANALYSIS")
print("=" * 40)

for camera in camera_data:
    score = min(int(camera["vehicle_density"] * 5), 100)

    if score >= 70:
        status = "🔴 SEVERE"
    elif score >= 40:
        status = "🟡 MODERATE"
    else:
        status = "🟢 LOW"

    print(
        f"{camera['camera_id']} → "
        f"Congestion Score: {score}/100 → {status}"
    )

In [ ]:
import matplotlib.pyplot as plt

camera_ids = [c["camera_id"] for c in camera_data]
vehicles = [c["total_vehicles"] for c in camera_data]

plt.figure(figsize=(10, 5))

bars = plt.bar(camera_ids, vehicles)

plt.title("🚦 City-Wide Traffic Monitoring")
plt.xlabel("Camera")
plt.ylabel("Number of Vehicles")

for bar, value in zip(bars, vehicles):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        str(value),
        ha="center",
        fontsize=12
    )

plt.ylim(0, max(vehicles) + 15)
plt.grid(axis="y", alpha=0.3)

plt.show()

In [ ]:
!pip install easyocr -q

In [ ]:
import easyocr

reader = easyocr.Reader(['en'])

print("🔢 ANPR OCR is ready!")

In [ ]:
import cv2

video_path = "/content/6420_India_Traffic_1280x720 (1).mp4"

cap = cv2.VideoCapture(video_path)

cap.set(cv2.CAP_PROP_POS_FRAMES, 100)

ret, frame = cap.read()
cap.release()

if ret:
    cv2.imwrite("/content/traffic_frame.jpg", frame)
    print("📸 Traffic frame extracted successfully!")
else:
    print("❌ Could not extract frame")

In [ ]:
import easyocr

image_path = "/content/traffic_frame.jpg"

results_ocr = reader.readtext(image_path)

print("🔢 ANPR RESULTS")
print("=" * 40)

if results_ocr:
    for detection in results_ocr:
        text = detection[1]
        confidence = detection[2]

        print(f"Detected: {text}")
        print(f"Confidence: {confidence:.2f}")
        print()
else:
    print("❌ No text detected")

In [ ]:
image_path = "/content/number_plate.webp"

results_ocr = reader.readtext(image_path)

print("🔢 ANPR RESULTS")
print("=" * 40)

if results_ocr:
    for detection in results_ocr:
        text = detection[1]
        confidence = detection[2]
        print(f"Detected: {text}")
        print(f"Confidence: {confidence:.2f}")
        print()
else:
    print("❌ No text detected")

In [ ]:
print("🚦 INTELLIGENT TRAFFIC MONITOR")
print("=" * 55)

# City-wide summary
total_city_vehicles = sum(c["total_vehicles"] for c in camera_data)

print(f"🏙️ Total Vehicles Detected: {total_city_vehicles}")
print()

# Camera status
for camera in camera_data:
    print(
        f"{camera['camera_id']} | "
        f"Vehicles: {camera['total_vehicles']} | "
        f"Traffic: {camera['traffic_level']} | "
        f"Density: {camera['vehicle_density']}"
    )

print()
print("🔢 ANPR")
print("-" * 55)

for detection in results_ocr:
    text = detection[1]
    confidence = detection[2]

    if text != "IND":
        print(f"Plate/Text: {text} | Confidence: {confidence:.2f}")

print()
print("🚨 ALERT SYSTEM")
print("-" * 55)

for camera in camera_data:
    if "HIGH" in camera["traffic_level"]:
        print(f"🚨 {camera['camera_id']}: HIGH TRAFFIC — ACTION REQUIRED")
    elif "MEDIUM" in camera["traffic_level"]:
        print(f"⚠️ {camera['camera_id']}: MODERATE TRAFFIC")
    else:
        print(f"✅ {camera['camera_id']}: TRAFFIC FLOWING NORMALLY")

print()
print("✅ AI TRAFFIC MONITORING COMPLETE")

In [ ]:
from IPython.display import display, HTML

display(HTML("""
<h1 style="text-align:center;">🚦 INTELLIGENT TRAFFIC MONITOR</h1>

<h3 style="text-align:center;">
AI-Powered City-Wide Traffic Analysis
</h3>

<hr>

<div style="display:flex; justify-content:space-around; text-align:center;">

<div>
<h2>🚗 156</h2>
<p>Total Vehicles</p>
</div>

<div>
<h2>🔴 1</h2>
<p>High Traffic</p>
</div>

<div>
<h2>🟡 1</h2>
<p>Medium Traffic</p>
</div>

<div>
<h2>🟢 1</h2>
<p>Low Traffic</p>
</div>

</div>

<hr>

<h2>📡 Camera Status</h2>

<table style="width:100%; text-align:center; font-size:18px;">
<tr>
<th>Camera</th>
<th>Vehicles</th>
<th>Traffic</th>
<th>Density</th>
</tr>

<tr>
<td>CAM_01</td>
<td>83</td>
<td>🔴 HIGH</td>
<td>12.97</td>
</tr>

<tr>
<td>CAM_02</td>
<td>52</td>
<td>🟡 MEDIUM</td>
<td>8.13</td>
</tr>

<tr>
<td>CAM_03</td>
<td>21</td>
<td>🟢 LOW</td>
<td>3.28</td>
</tr>

</table>

<hr>

<h2>🚨 Traffic Alerts</h2>

<p style="font-size:18px;">
🚨 CAM_01 — HIGH TRAFFIC — ACTION REQUIRED
</p>

<p style="font-size:18px;">
⚠️ CAM_02 — MODERATE TRAFFIC
</p>

<p style="font-size:18px;">
✅ CAM_03 — TRAFFIC FLOWING NORMALLY
</p>

<hr>

<h2>🔢 ANPR</h2>

<p style="font-size:18px;">
Detected Text: <b>Ka99</b> — Confidence: 53%
</p>

<p style="font-size:18px;">
Detected Text: <b>E011316</b> — Confidence: 68%
</p>

<hr>

<h3 style="text-align:center;">
✅ AI TRAFFIC MONITORING SYSTEM ACTIVE
</h3>
"""))

In [ ]:
from ultralytics import YOLO
import cv2
import glob
from IPython.display import Video, display

# Run ByteTrack
track_results = model.track(
    source=next(iter(uploaded)),
    tracker="bytetrack.yaml",
    save=True,
    conf=0.25
)

# Find the generated video
avi_files = glob.glob("/content/runs/detect/track/*.avi")

print("Tracking completed ✅")
print("Output files:", avi_files)

if avi_files:
    input_avi = avi_files[-1]
    output_mp4 = "/content/bytetrack_final.mp4"

    # Convert AVI → MP4
    cap = cv2.VideoCapture(input_avi)

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(output_mp4, fourcc, fps, (width, height))

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        out.write(frame)

    cap.release()
    out.release()

    print("🎥 ByteTrack video created!")
    print(output_mp4)

    display(Video(output_mp4, embed=True))
else:
    print("❌ Tracking video was not found")

In [ ]:
import os
import glob

print("📁 Checking YOLO output files...")

files_found = glob.glob("/content/runs/detect/predict/*")

for f in files_found:
    print("✅", f)

In [ ]:
import subprocess
from IPython.display import Video, display

input_avi = "/content/runs/detect/predict/6420_India_Traffic_1280x720 (1).avi"
output_mp4 = "/content/bytetrack_final.mp4"

subprocess.run([
    "ffmpeg", "-y",
    "-i", input_avi,
    "-c:v", "libx264",
    "-pix_fmt", "yuv420p",
    output_mp4
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("🎥 ByteTrack final video created!")
print(output_mp4)

display(Video(output_mp4, embed=True))

In [ ]:
from IPython.display import Image, display

print("🔢 Number Plate Used for ANPR")
display(Image(filename="/content/number_plate.webp"))

In [ ]:
import os

print(os.listdir("/content"))

In [ ]:
import os
print(os.listdir("/content"))

In [ ]:
from IPython.display import Image, display

print("🔢 NUMBER PLATE USED FOR ANPR")
display(Image(filename="/content/number_plate.webp"))


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

img = Image.open("/content/number_plate.webp")

plt.figure(figsize=(10, 4))
plt.imshow(img)
plt.axis("off")
plt.title("🔢 Number Plate Used for ANPR")
plt.show()

In [ ]:
results_ocr = reader.readtext("/content/number_plate.webp")

print("🔢 ANPR RESULTS")
print("=" * 40)

for detection in results_ocr:
    text = detection[1]
    confidence = detection[2]

    print(f"Detected: {text}")
    print(f"Confidence: {confidence:.2f}")
    print()

In [ ]:
print(reader)

In [ ]:
import easyocr

reader = easyocr.Reader(['en'])

print("🔢 ANPR OCR is ready! 🚗")

In [ ]:
!pip install easyocr -q